In [ ]:
from dotenv import load_dotenv
from pathlib import Path
import os
import requests
import json
import time
from datetime import datetime, timedelta, timezone, time as dt_time
from dateutil.relativedelta import relativedelta

In [ ]:
class TokenManager:
    def __init__(self, server, token_file, refresh_file):
        self.server = server
        self.token_file = token_file
        self.refresh_file = refresh_file
        self.token = None
    
    # --------------------------
    # Main Entry: Return Token
    # --------------------------

    def get_valid_token(self):
        self.token = self._load_token()
        if self.token and self._check_token_validity(self.token):
            print('Token valid')
            return self.token

        print('Token invalid try to refresh ...')
        refreshed = self._refresh_token()
        if refreshed:
            return refreshed

        raise SystemExit('Both token and refresh failed') 

    # ---------------
    # Load Token
    # ---------------

    def _load_token(self):
        if os.path.exists(self.token_file):
            print(f"Checking token file at: {self.token_file}")
            with open(self.token_file, 'r') as f:
                token = f.read().strip()
                if token:
                    print('Access token loaded')
                    return token
        print ('No token found')
        return None
    
    # --------------------
    # Load Refresh Token
    # --------------------

    def _refresh_token(self):
        refresh_api =  f'{self.server}/api/auth/token'

        if not os.path.exists(self.refresh_file):
            print('No refresh token')
            return None
        with open(self.refresh_file, 'r') as f:
            refresh = f.read().strip()
            print('Refresh token loaded')
        if not refresh:
            print('Refresh token empty')
            return None
        
        try:
            response = requests.post(refresh_api, json={'refreshToken': refresh})
            
            if response.status_code == 200:
                data = response.json()
                new_token = data.get('token')

                if new_token:
                    with open(self.token_file,'w') as f:
                        f.write(new_token)
                    print('Access token refreshed')
                    self.token = new_token
                    return new_token
                
                print('Refresh succeed but No Token')
                return None
            
            print(f'Refresh Failed: {response.status_code} - {response.text}')
            return None

        except Exception as e:
            print(f'Error during refresh {e}')
            return None
    
    # ---------------------
    # Check Validity Token
    # ---------------------

    def _check_token_validity(self, token):
        url = f'{self.server}/api/auth/user'
        headers = {'X-Authorization':f'Bearer {token}'}

        try:
            r = requests.get(url, headers=headers)
            return r.status_code == 200
        except:
            return False

# ----------------------
# Get Directory for Env
# ----------------------

base_dir = Path(os.getcwd()).resolve().parent

load_dotenv(base_dir / ".env")

manager = TokenManager(
    server=os.getenv("SERVER"),
    token_file=str(base_dir/os.getenv("TOKEN")),
    refresh_file=str(base_dir/os.getenv("REFRESH"))
)

token = manager.get_valid_token()

headers = {
    "Accept": "application/json",
    "X-Authorization": f"Bearer {token}"
}

In [ ]:
obj = ['mobil_pribadi','truk','pickup']
keys = []
for obj_view in obj:
    keys.append(f'{obj_view}_in2_delta')
    keys.append(f'{obj_view}_out_delta')
    keys.append(f'{obj_view}_turn_delta')

# Configuration 
CONFIG = {
    "server": os.getenv("SERVER"),
    "asset_name": "VTC-DEMO-CAM1",
    "asset_id": os.getenv("ASSET_ID"),
    "asset_token": os.getenv("ASSET_TOKEN"),
    "keys": ",".join(keys),
    "start_collect": "07/05/2025",
    "end_collect": (datetime.now() - timedelta(days=0)).strftime("%m/%d/%Y"),
    "interval": 2,
    "agg": "SUM"
}

In [ ]:
# ----------------------
# Generate Date Ranges
# ----------------------

start_date = datetime.strptime(CONFIG['start_collect'], "%m/%d/%Y")
end_date = datetime.strptime(CONFIG['end_collect'], "%m/%d/%Y")

ranges = []
current_start = start_date
while current_start <= end_date:
    current_end = min(current_start + relativedelta(months=1) - timedelta(days=0), end_date)
    ranges.append((
        current_start.strftime("%m/%d/%Y"),
        current_end.strftime("%m/%d/%Y")
        ))
    current_start = current_start + relativedelta(months=1)

print(f'Data range will be split into {len(ranges)} parts: ')

for i, (start, end) in enumerate(ranges, 1):
    print(f'Part {i}: {start} to {end}')

batch = 2

def chunk_list(data, size):
    for i in range(0, len(data), size):
        yield data[i: i + size]

range_batch = list(chunk_list(ranges, batch))

print(f'Total batches:{len(range_batch)}')

In [ ]:
# ------------
# Fetch Data
# ------------

telemetry_api = f"{CONFIG['server']}/api/plugins/telemetry/DEVICE/{CONFIG['asset_id']}/values/timeseries"
headers = {"Accept": "application/json", "X-Authorization": f"Bearer {token}"}

def convert_date_to_ts(date):
    return int(datetime.strptime(date, "%m/%d/%Y").timestamp() * 1000)

keys_list = CONFIG['keys'].split(',')

start_batch = 3
max_retry = 3
for batch_to_run in range(start_batch, len(range_batch) + 1):

    current_batch = range_batch[batch_to_run -1]
    all_data = {}

    print(f'Running batch {batch_to_run} with {len(current_batch)} parts')

    for i, (start_date_str,  end_date_str) in enumerate(current_batch, 1):
        print(f'Fetching data for range : {start_date_str} to {end_date_str}')
    
        for key in keys_list:
            params = {
                "keys": key,
                "startTs": convert_date_to_ts(start_date_str),
                "endTs": convert_date_to_ts(end_date_str),
                "interval": int(CONFIG["interval"] * 60 * 60 * 1000),
                "agg": CONFIG["agg"],
                "useStrictDataTypes": "false"
            }
            
            range_data = None

            for attempt in range(1, max_retry + 1):
                try:
                    response = requests.get(telemetry_api, headers=headers, params=params, timeout=30)

                    if response.status_code == 200:
                        range_data = response.json()
                        break

                    print(
                        f'Attempt{attempt}/{max_retry} failed'
                        f'({response.status_code}): {response.text}' 
                    )

                except requests.exceptions.RequestException as e:
                    print(f"Attempt {attempt}/{max_retry} exception: {e}")
                    
                time.sleep(attempt * 2)

            if not range_data:
                raise SystemExit

            all_data.setdefault(key, []).extend(range_data[key])
        time.sleep(1)
    
    output_path = base_dir/"data"/"raw"/f"batch_{batch_to_run}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w') as f:
        json.dump(all_data,f)

    print(f'Batch {batch_to_run} completed')